# Exercise: Extend and Evaluate a LangChain RAG Pipeline

In the lesson you built a RAG pipeline with LangChain. You start from the working pipeline and take it further: make its answers **traceable** by returning their sources, **tune** how much context it retrieves, and then **evaluate** the result so you can say which setup is better, not just guess.

For the concepts behind evaluation, see [RAG Concepts](../1_rag_concepts.md).

## Learning Objectives

At the end of this notebook, you should be able to:

- Extend a RAG pipeline to return the source passages behind each answer.
- Tune retrieval (the number of chunks) and observe its effect on the answer.
- Evaluate and compare two configurations with an LLM-as-judge (faithfulness and answer relevance).

## Provided: the pipeline from the lesson

Run this cell as is. It gives you `build_chain(pdf_path, k=..., chunk_size=..., chunk_overlap=...)`, which returns a ready retrieval chain, and `ask(question, chain)`, which invokes a chain and returns the full result (the `input`, the retrieved `context`, and the `answer`). It also builds `react_chain` over the React paper for you. You do not need to change anything here.

In [ ]:
import warnings

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client
from pypdf import PdfReader

In [ ]:
load_dotenv()
warnings.filterwarnings("ignore")

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

In [ ]:
def build_chain(pdf_path, k=4, chunk_size=200, chunk_overlap=50):
    """Load a PDF, build a Chroma retriever, and wire it to the LLM. Returns a retrieval chain."""
    reader = PdfReader(pdf_path)
    docs = [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": pdf_path, "page": i},
        )
        for i, page in enumerate(reader.pages, start=1)
    ]
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    ).split_documents(docs)
    embedding = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2",
        encode_kwargs={"normalize_embeddings": True},
    )
    store = Chroma.from_documents(
        chunks, embedding, collection_metadata={"hnsw:space": "cosine"}
    )
    retriever = store.as_retriever(search_kwargs={"k": k})
    prompt = Client().pull_prompt(
        "langchain-ai/retrieval-qa-chat",
        include_model=False,
        dangerously_pull_public_prompt=True,
    )
    combine = create_stuff_documents_chain(llm=llm, prompt=prompt)
    return create_retrieval_chain(retriever=retriever, combine_docs_chain=combine)


def ask(question, chain):
    """Invoke a chain and return the full result dict (input, context, answer)."""
    return chain.invoke({"input": question})


# Build the pipeline over the React paper, ready to use
react_chain = build_chain("../documents/react_paper.pdf")

## Exercise 1: Make answers traceable with source citations

A RAG answer is only trustworthy if you can check where it came from. The result from `ask` includes a `context` list of the retrieved documents, and each document carries metadata, including its page number in `doc.metadata`.

Write a function that asks a question and returns the answer **together with the source pages** of the chunks that supported it.

Hints:

- Call `ask(question, react_chain)` and read `result["answer"]` and `result["context"]`.
- Each item in `context` is a `Document`; look at `doc.metadata` to find the page.

In [ ]:
def answer_with_sources(question):
    """Return the answer plus the source pages of the retrieved chunks."""
    result = ask(question, react_chain)
    pages = sorted({doc.metadata.get("page") for doc in result["context"]})
    return result["answer"], pages

**Explanation**

The chain result includes a `context` list of `Document` objects, each carrying metadata such as the page number. Collecting `doc.metadata.get('page')` lets the answer cite where it came from, which is what makes a RAG answer auditable.

In [ ]:
answer, pages = answer_with_sources("What is the ReAct framework?")
print("source pages:", pages)
print(answer)

## Exercise 2: Tune how much context you retrieve

The retriever returns the top `k` chunks. Too few may miss the answer, too many may dilute the prompt with irrelevant text. `build_chain` accepts a `k` argument, so you can build chains with different values.

Build two chains over the React paper, one with a small `k` and one with a larger `k`, ask both the same question, and compare the answers.

Hint: `chain_small = build_chain("../documents/react_paper.pdf", k=2)`.

In [ ]:
chain_small = build_chain("../documents/react_paper.pdf", k=2)
chain_large = build_chain("../documents/react_paper.pdf", k=6)

question = "What is the ReAct framework?"
print("k=2:\n", ask(question, chain_small)["answer"])
print("\nk=6:\n", ask(question, chain_large)["answer"])

## Exercise 3: Evaluate and compare the two configurations

Comparing answers by eye does not scale. Use the model as a judge to score each answer, then let the scores decide which `k` worked better.

Write a small `llm_judge` function that asks the model to rate something from 1 to 5, then use it to score two things for each answer:

- **Faithfulness**: is the answer supported by the retrieved context?
- **Answer relevance**: does the answer address the question?

Compare the scores for your small-`k` and large-`k` chains, and write a sentence on which you would choose and why.

Hint: build the judging prompt from the question, the context, and the answer, send it with the provided `llm`, and parse a single digit from the reply.

In [ ]:
import re


def llm_judge(instruction):
    """Ask the model to rate something from 1 to 5 and return the number."""
    response = llm.invoke(instruction + "\n\nReply with a single integer from 1 to 5.")
    content = response.content
    if not isinstance(content, str):
        return None
    match = re.search(r"[1-5]", content)
    return int(match.group()) if match else None

In [ ]:
def evaluate(chain, question):
    result = ask(question, chain)
    context = "\n---\n".join(doc.page_content for doc in result["context"])
    faithfulness = llm_judge(
        f"How well is this answer supported by the context?\n\n"
        f"Context:\n{context}\n\nAnswer: {result['answer']}"
    )
    relevance = llm_judge(
        f"How well does this answer address the question?\n\n"
        f"Question: {question}\n\nAnswer: {result['answer']}"
    )
    return faithfulness, relevance


question = "What is the ReAct framework?"
print("k=2 (faithfulness, relevance):", evaluate(chain_small, question))
print("k=6 (faithfulness, relevance):", evaluate(chain_large, question))

## Conclusion and Reflection

**Did citations change how much you trust the answers? When would source pages matter most?**

Citations make an answer auditable: you can open the cited page and confirm the claim, which raises trust. They matter most in high-stakes or regulated settings (medical, legal, financial) and whenever users need to verify a claim rather than take the model's word.

**Which k gave better judge scores, and did that match your own reading of the answers?**

A larger k supplies more supporting context, which often raises the faithfulness score, but it can also add irrelevant passages that lower relevance. There is no universally best k: compare the scores for your two settings and confirm by reading the answers, since the judge is only an approximation.

**The judge is itself an LLM and can be biased. How would you make the evaluation more reliable, and where would a tool like RAGAS or a labelled test set help?**

Make it more reliable by averaging several judgements, using a stronger judge model, fixing a clear scoring rubric, and validating the judge against a small set of human-labelled examples. A labelled test set lets you measure precision, recall, and answer correctness directly, and RAGAS standardises the judge-based metrics so results are comparable across runs.